In [ ]:
!pip install transformers datasets evaluate numpy torch sentencepiece nltk rouge_score accelerate>=0.26.0
!pip install PyDrive
!pip install google-auth-oauthlib


In [ ]:
import os
import json
from glob import glob
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import evaluate
from evaluate import load
import zipfile
import os
import evaluate
from evaluate import load



In [ ]:
import torch

print(torch.cuda.is_available())

In [ ]:
zip_path = ""    
extract_dir = "data_jsons"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Unpacked'{extract_dir}'")


In [ ]:
MODEL_NAME = "SGaleshchuk/t5-large-ua-news"
DATA_DIR = "data_jsons/summaries_short_v-3"
OUTPUT_DIR = "output_mt5_finetuned"

LEARNING_RATE = 1e-5
BATCH_SIZE = 1
GRAD_ACCUM = 8
NUM_TRAIN_EPOCHS = 5
FP16 = False
SEED = 42

WARMUP_STEPS = 500


In [ ]:
examples = []

for path in glob(os.path.join(DATA_DIR, "*")):
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        for chunk in data["chunks"]:
            text = chunk.get("text")
            summary = chunk.get("summary")

            if text and summary:
                examples.append({
                    "text": text,
                    "summary": summary
                })

    except Exception as e:
        print("skip", path, "err", e)

print(f"Loaded {len(examples)} examples")

ds = Dataset.from_list(examples)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)


# Preprocessing

In [ ]:
def check_lengths(dataset, tokenizer, max_samples=None, print_examples=32):
    import numpy as np

    article_lengths = []
    summary_lengths = []

    long_articles = []
    long_summaries = []

    for i, item in enumerate(dataset):
        if max_samples and i >= max_samples:
            break

        article = item["text"]
        summary = item["summary"]

        art_tokens = tokenizer(
            article,
            truncation=False,
            add_special_tokens=False
        )["input_ids"]

        sum_tokens = tokenizer(
            summary,
            truncation=False,
            add_special_tokens=False
        )["input_ids"]

        art_len = len(art_tokens)
        sum_len = len(sum_tokens)

        article_lengths.append(art_len)
        summary_lengths.append(sum_len)

        if art_len > 512 and len(long_articles) < print_examples:
            long_articles.append((art_len, article))

        if sum_len > 512 and len(long_summaries) < print_examples:
            long_summaries.append((sum_len, summary))

    def print_stats(name, arr):
        print(f"\n===== {name.upper()} =====")
        print("Checked:", len(arr))
        print("Max length:", arr.max())
        print("Mean:", arr.mean())
        print("Median:", np.median(arr))

        for threshold in [160, 180, 200, 220, 240, 512]:
            count = (arr > threshold).sum()
            percent = (arr > threshold).mean() * 100
            print(f"Count >{threshold}: {count}")
            print(f"Percent >{threshold}: {percent:.2f}%")

    article_arr = np.array(article_lengths)
    summary_arr = np.array(summary_lengths)

    print_stats("text", article_arr)
    print_stats("summary", summary_arr)
    

check_lengths(examples, tokenizer, max_samples=15000)

In [ ]:
def filter_long_articles(dataset, tokenizer, max_len=512):
    def is_valid(example):
        tokens = tokenizer(
            example["text"],
            truncation=False,
            add_special_tokens=False
        )["input_ids"]
        return len(tokens) <= max_len

    filtered = dataset.filter(is_valid)

    print(f"Original: {len(dataset)}")
    print(f"Filtered: {len(filtered)}")
    print(f"Removed: {len(dataset) - len(filtered)}")

    return filtered


filtered_ds = filter_long_articles(ds, tokenizer)

In [ ]:
def preprocess(batch):
    model_inputs = tokenizer(
        batch["text"],
        truncation=False
    )

    labels = tokenizer(
        text_target=batch["summary"],
        truncation=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = filtered_ds.map(preprocess, batched=True, remove_columns=filtered_ds.column_names)

In [ ]:
split = tokenized.train_test_split(test_size=0.2, seed=SEED)
test_valid = split["test"].train_test_split(test_size=0.5, seed=SEED)

dataset_dict = {
    "train": split["train"],
    "validation": test_valid["train"],
    "test": test_valid["test"]
}

print(dataset_dict)


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",
    label_pad_token_id=-100
)


In [ ]:
rouge = load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    return rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )



In [ ]:
train_subset = dataset_dict["train"].shuffle(seed=SEED)
val_subset = dataset_dict["validation"].shuffle(seed=SEED)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    save_strategy="no", 
    predict_with_generate=True,
    eval_strategy="epoch",

    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    fp16=FP16,

    logging_steps=100,
    seed=SEED,
    report_to="none",
)


In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=val_subset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)



model.gradient_checkpointing_enable()

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
print("Model saved to", OUTPUT_DIR)